# Micronuclei Frequency and IRF3 Nuclear Signal in MN+ vs MN− Cells

Segments nuclei from Sytox fluorescence images and quantifies micronuclei frequency alongside IRF3 nuclear signal, stratifying cells by micronucleus status (MN+ vs MN−). Micronuclei are detected in the perinuclear region and filtered by morphology and IRF3 intensity. Results are exported to CSV for downstream statistical analysis.

## 1. Dependencies

In [ ]:
import os
import re
import cv2
import numpy as np
import pandas as pd
from skimage.feature import peak_local_max

## 2. Configuration

In [ ]:
# --- Input ---
directory = "image3_npy"  # Input folder; each .npy file is a (2, H, W) array: [IRF3, Sytox]

# --- Group mapping ---
group_map = {
    '25': 'Mock_32H_Rep1',
    '26': 'GS_32H_Rep1',
    '27': 'Mock_32H_Rep2',
    '28': 'GS_32H_Rep2'
}
order = ['25','26','27','28']

# --- Nucleus segmentation parameters ---
global_blur_size = (35, 35)  # Gaussian blur size for Sytox before thresholding
specific_blur_size = (65, 65)   # Blur within each ROI to refine nuclear mask
min_nucleus_area = 3500  # Minimum valid nucleus area (pixels)
circularity_cutoff = 0.65  # Shape filter to exclude elongated or fragmented objects (established range: 0.6–0.7)
pad_px = 100  # Padding around each nucleus bounding box for micronuclei search (pixels)
micronuclei_area_range = (25, 1000)  # Valid area range for micronuclei candidates (pixels)
micronuclei_circ_cutoff = 0.65  # Circularity threshold for micronuclei candidates
rescue_threshold = -20  # Percent-difference cutoff for classification of nuclear IRF3 localization
mn_intensity_cutoff = 35  # Minimum mean IRF3 intensity required for a valid micronucleus candidate

## 3. Image Processing Pipeline

In [ ]:
# ===============================================================
# INITIALIZE STORAGE
# ===============================================================
delta_by_group = {k: [] for k in group_map}
results_by_group = {k: [] for k in group_map}
labels_by_group  = {k: [] for k in group_map}

# ===============================================================
# IMAGE PROCESSING LOOP
# ===============================================================
npy_files = [f for f in os.listdir(directory) if f.endswith(".npy")]
all_areas = []
circularities = []

for filename in sorted(npy_files):
    prefix = re.match(r"\d+", filename).group()
    if prefix not in group_map:
        continue
    condition = group_map[prefix]
    data = np.load(os.path.join(directory, filename))

    if data.shape[0] < 2:
        print(f"[warn] {filename}: missing channels")
        continue

    irf3_gray, sytox_gray = data[0].astype(np.uint8), data[1].astype(np.uint8)
    height, width = sytox_gray.shape

    # --- Segment nuclei from Sytox ---
    blurred = cv2.GaussianBlur(sytox_gray, global_blur_size, 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    areas = [cv2.contourArea(cnt) for cnt in contours]
    all_areas.extend(areas)

    def is_inside(cnt):
        x, y, w, h = cv2.boundingRect(cnt)
        return x > 0 and y > 0 and x + w < width and y + h < height

    filtered = [c for c in contours if cv2.contourArea(c) >= min_nucleus_area and is_inside(c)]

    # --- Per-nucleus analysis ---
    for j, cnt in enumerate(filtered):
        x, y, w, h = cv2.boundingRect(cnt)
        x1, y1 = max(x - pad_px, 0), max(y - pad_px, 0)
        x2, y2 = min(x + w + pad_px, width), min(y + h + pad_px, height)

        roi_irf3, roi_sytox = irf3_gray[y1:y2, x1:x2], sytox_gray[y1:y2, x1:x2]
        nucleus_roi = cnt - [x1, y1]

        # Secondary blur refines the nuclear boundary within the cropped ROI
        roi_blur = cv2.GaussianBlur(roi_sytox, specific_blur_size, 0)
        _, roi_thr = cv2.threshold(roi_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        roi_cnts, _ = cv2.findContours(roi_thr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not roi_cnts:
            continue

        mask_global = np.zeros_like(roi_sytox, np.uint8)
        cv2.drawContours(mask_global, [nucleus_roi], -1, 255, -1)
        best, overlap = None, 0
        for rc in roi_cnts:
            mask_rc = np.zeros_like(roi_sytox, np.uint8)
            cv2.drawContours(mask_rc, [rc], -1, 255, -1)
            ov = cv2.countNonZero(cv2.bitwise_and(mask_global, mask_rc))
            if ov > overlap:
                best, overlap = rc, ov
        if best is None:
            continue

        area = cv2.contourArea(best)
        perim = cv2.arcLength(best, True)
        circ = 4 * np.pi * area / (perim ** 2) if perim > 0 else 0
        circularities.append(circ)
        if area < min_nucleus_area or circ < circularity_cutoff:
            continue

        # --- Build masks ---
        mask_in = np.zeros_like(roi_sytox, np.uint8)
        cv2.drawContours(mask_in, [best], -1, 255, -1)
        k_in = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
        k_out = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (125, 125))
        ring_mask = cv2.subtract(cv2.dilate(mask_in, k_out, 1), cv2.dilate(mask_in, k_in, 1))

        # --- Build an outer-edge "no-go" band and a safe ring (inner-edge touches allowed) ---
        edge_clearance_px = 1  # 0 = only exclude exact boundary pixels; 1-2 gives a small margin

        outer_mask = cv2.dilate(mask_in, k_out, 1)  # full area up to the outer ring
        kernel_clear = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                                 (2*edge_clearance_px + 1, 2*edge_clearance_px + 1))

        # Erode from outside to make a safe outer area; the difference is the "no-go" band along the outer edge
        safe_outer = cv2.erode(outer_mask, kernel_clear, 1)
        unsafe_outer_band = cv2.subtract(outer_mask, safe_outer)
        safe_ring = cv2.bitwise_and(ring_mask, cv2.bitwise_not(unsafe_outer_band))

        # --- Micronuclei detection (with containment + intensity filters) ---
        ring_sytox = cv2.bitwise_and(roi_sytox, roi_sytox, mask=ring_mask)
        _, ring_thr = cv2.threshold(ring_sytox, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        ring_cnts, _ = cv2.findContours(ring_thr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        mn = []
        for rc in ring_cnts:
            a = cv2.contourArea(rc)
            p = cv2.arcLength(rc, True)
            if p == 0:
                continue
            c = 4 * np.pi * (a / (p ** 2))
            if not (micronuclei_area_range[0] < a < micronuclei_area_range[1] and c > micronuclei_circ_cutoff):
                continue

            mn_mask = np.zeros_like(roi_sytox, np.uint8)
            cv2.drawContours(mn_mask, [rc], -1, 255, -1)
            overlap = cv2.bitwise_and(mn_mask, ring_mask)

            candidate_intensity = cv2.mean(roi_irf3, mask=mn_mask)[0]

            # --- Require micronucleus fully inside the safe ring (may touch inner edge, not outer) ---
            if not np.array_equal(cv2.bitwise_and(mn_mask, safe_ring), mn_mask):
                continue

            # --- Require sufficient IRF3 intensity ---
            if candidate_intensity < mn_intensity_cutoff:
                continue

            mn.append(rc)

        is_mn = len(mn) > 0
        mn_area_total = sum(cv2.contourArea(rc) for rc in mn) if is_mn else 0

        # Compute mean IRF3 intensity for valid micronuclei
        mn_intensities = []
        for rc in mn:
            mn_mask = np.zeros_like(roi_irf3, np.uint8)
            cv2.drawContours(mn_mask, [rc], -1, 255, -1)
            mean_mn_irf3 = cv2.mean(roi_irf3, mask=mn_mask)[0]
            mn_intensities.append(mean_mn_irf3)
        mean_mn_intensity = np.mean(mn_intensities) if mn_intensities else 0

        # --- IRF3 intensity metrics ---
        bg_mask = (roi_irf3 > 10).astype(np.uint8) * 255
        cyto_mask = cv2.bitwise_and(ring_mask, bg_mask)
        mean_in, mean_cyto = cv2.mean(roi_irf3, mask_in)[0], cv2.mean(roi_irf3, cyto_mask)[0]
        if mean_cyto == 0:
            continue
        delta = mean_in - mean_cyto
        perc_diff = ((mean_in - mean_cyto) / mean_cyto) * 100

        classification = "nuclear" if delta > 0 else "non-nuclear"
        if is_mn and classification == "nuclear":
            lbl = "MN+ nuclear"
        elif is_mn and classification == "non-nuclear":
            lbl = "MN+ non-nuclear"
        elif not is_mn and classification == "nuclear":
            lbl = "MN- nuclear"
        else:
            lbl = "MN- non-nuclear"

        results_by_group[prefix].append({
            "label": f"{filename}_cell{j}",
            "is_mn": is_mn,
            "label_group": lbl,
            "mean_inside": mean_in,
            "total_inside": float(cv2.sumElems(cv2.bitwise_and(roi_irf3, roi_irf3, mask=mask_in))[0]),
            "mean_cytoring": mean_cyto,
            "total_cytoring": float(cv2.sumElems(cv2.bitwise_and(roi_irf3, roi_irf3, mask=cyto_mask))[0]),
            "delta": delta,
            "perc_diff_mean": perc_diff,
            "area": area,
            "mn_area_total": mn_area_total,
            "mean_mn_intensity": mean_mn_intensity
        })
        labels_by_group[prefix].append(lbl)

## 4. Export Results

In [ ]:
# ===============================================================
# Save Data
# ===============================================================
rows = []
for k in order:
    cond = group_map[k]
    for r in results_by_group[k]:
        cls = "nuclear" if r["perc_diff_mean"] >= rescue_threshold else "non-nuclear"
        rows.append({
            "label": r["label"],
            "is_mn": r["is_mn"],
            "irf3_mean": r["mean_inside"],
            "irf3_total": r["total_inside"],
            "percent_diff": r["perc_diff_mean"],
            "condition": cond,
            "classification": cls,
            "area": r["area"],
            "mn_area_total": r["mn_area_total"]
        })

df = pd.DataFrame(rows)
df.to_csv("image3_resuls.csv",index=False)